<a href="https://colab.research.google.com/github/jsemprini/Iowa-Water-Nitrate-Births-8488update/blob/main/Copy_of_2_createbirth_final_T1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# File 2 (Final V3) — Create the gestational-start birth cohort and T1 window

**Purpose.** Prepare the 1983–1988 Iowa natality records for linkage to the completed county-quarter nitrate panel.

The final production pipeline retains the V3 gestational-start cohort logic but creates only one exposure window: **T1, gestational days 0–97 (98 days total)**.

### Cohort logic

- Begin with births delivered during 1983–1988.
- Reconstruct gestational start using the V3 LMP/detailed-gestational-age hierarchy.
- Require gestational start on/after 1983-01-01.
- Apply an outcome-independent right-edge rule: gestational start must be early enough to permit follow-up through 41w6d before the birth data end.
- Require an Iowa county represented in the water panel, singleton birth, and gestational age 20–41 completed weeks.
- Require the complete T1 window to fall inside the water panel.
- Retain all maternal race groups; race is carried forward for later analytic restriction or adjustment.

**Note: This code requires appended birth records (1983-1988) in current working directory accessed from NBER https://www.nber.org/research/data/vital-statistics-natality-birth-data**

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
pd.set_option('display.max_columns', 250)
pd.set_option('display.width', 260)


Mounted at /content/drive


In [ ]:
# ============================================================
# 0. PATHS AND STUDY SETTINGS
# ============================================================
RAW_BIRTH_PATH = (
    '/content/drive/MyDrive/Current Research/Water/birth-raw/'
    'natalityus1970-1988.csv'
)
FIPS_PATH = (
    '/content/drive/MyDrive/Current Research/Water/water-raw/fips.csv'
)
WATER_PANEL_PATH = (
    '/content/drive/MyDrive/plos-update-v3/1-water/'
    'final_county_quarter_complete_1982_1988_v3.csv'
)
OUTPUT_DIR = '/content/drive/MyDrive/plos-update-v3/2-birth'
os.makedirs(OUTPUT_DIR, exist_ok=True)

STUDY_GEST_START = pd.Timestamp('1983-01-01')
PRIMARY_GEST_MIN_WEEKS = 20
PRIMARY_GEST_MAX_COMPLETED_WEEKS = 41
# dgestat is completed weeks. 41 includes 41w0d through 41w6d.
MAX_FOLLOWUP_DAYS = (PRIMARY_GEST_MAX_COMPLETED_WEEKS + 1) * 7 - 1  # 293 days
T1_END_DAY = 97  # inclusive; gestational days 0-97 = 98 days

for label, path in {
    'Raw natality file': RAW_BIRTH_PATH,
    'County crosswalk': FIPS_PATH,
    'Water panel': WATER_PANEL_PATH,
}.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f'{label} not found:\n{path}')
    print('FOUND:', label, '->', path)


FOUND: Raw natality file -> /content/drive/MyDrive/Current Research/Water/birth-raw/natalityus1970-1988.csv
FOUND: County crosswalk -> /content/drive/MyDrive/Current Research/Water/water-raw/fips.csv
FOUND: Water panel -> /content/drive/MyDrive/plos-update-v3/1-water/final_county_quarter_complete_1982_1988_v3.csv


In [ ]:
# ============================================================
# 1. LOAD 1983-1988 BIRTH RECORDS
# ============================================================
# Copy the large natality CSV to local Colab storage before reading. This avoids
# intermittent Google Drive transport failures during a long pd.read_csv call.
LOCAL_BIRTH_PATH = '/content/natalityus1970-1988.csv'
if not os.path.exists(LOCAL_BIRTH_PATH):
    print('Copying natality file from Google Drive to local Colab storage...')
    shutil.copy2(RAW_BIRTH_PATH, LOCAL_BIRTH_PATH)

print('Reading natality file from:', LOCAL_BIRTH_PATH)
df_raw = pd.read_csv(LOCAL_BIRTH_PATH, low_memory=False)
df_raw['birth_id'] = np.arange(1, len(df_raw) + 1, dtype=np.int64)

def numeric_column(frame, variable):
    if variable in frame.columns:
        return pd.to_numeric(frame[variable], errors='coerce')
    return pd.Series(np.nan, index=frame.index, dtype='float64')

core_required = [
    'yr', 'birmon', 'birday', 'cntyres', 'dgestat', 'dbirwt', 'dplural',
    'dmage', 'dlivord', 'dmeduc', 'mrace', 'csex', 'monprec', 'dmar',
    'lmpmon', 'lmpday', 'lmpyr'
]
missing_core = [c for c in core_required if c not in df_raw.columns]
if missing_core:
    raise KeyError('Required variables missing: ' + ', '.join(missing_core))

df_raw['birth_year'] = numeric_column(df_raw, 'yr')
df = df_raw.loc[df_raw['birth_year'].between(1983, 1988)].copy()
print('Births delivered 1983-1988:', f'{len(df):,}')


Copying natality file from Google Drive to local Colab storage...
Reading natality file from: /content/natalityus1970-1988.csv
Births delivered 1983-1988: 241,633


In [ ]:
# ============================================================
# 2. BIRTH DATE, GESTATIONAL AGE, AND LMP/DGESTAT START HIERARCHY
# ============================================================
df['birth_month'] = numeric_column(df, 'birmon')
df['birth_day'] = numeric_column(df, 'birday')
df['birth_date'] = pd.to_datetime({
    'year': df['birth_year'], 'month': df['birth_month'], 'day': df['birth_day']
}, errors='coerce')
df['birth_date_valid'] = df['birth_date'].notna().astype(np.int64)

# Keep the NCHS edited range for reconstruction; final cohort plausibility is 20-41.
df['dgestat_raw'] = numeric_column(df, 'dgestat')
df['gest_age_weeks'] = df['dgestat_raw'].where(df['dgestat_raw'].between(17, 52))
df['gest_age_valid'] = df['gest_age_weeks'].notna().astype(np.int64)
df['gestation_start_dgestat'] = (
    df['birth_date'] - pd.to_timedelta(df['gest_age_weeks'] * 7, unit='D')
)

# Exact LMP reconstruction.
df['lmp_month_raw'] = numeric_column(df, 'lmpmon')
df['lmp_day_raw'] = numeric_column(df, 'lmpday')
df['lmp_year_raw'] = numeric_column(df, 'lmpyr')
df['lmp_year_full'] = np.nan
m4 = df['lmp_year_raw'].between(1900, 1999)
df.loc[m4, 'lmp_year_full'] = df.loc[m4, 'lmp_year_raw']
m2 = df['lmp_year_raw'].between(10, 98)
df.loc[m2, 'lmp_year_full'] = 1900 + df.loc[m2, 'lmp_year_raw']
m1 = df['lmp_year_raw'].between(0, 9)
same_year = m1 & ((df['birth_year'] % 10) == df['lmp_year_raw'])
prev_year = m1 & (((df['birth_year'] - 1) % 10) == df['lmp_year_raw'])
df.loc[same_year, 'lmp_year_full'] = df.loc[same_year, 'birth_year']
df.loc[prev_year, 'lmp_year_full'] = df.loc[prev_year, 'birth_year'] - 1

lmp_components = pd.DataFrame({
    'year': pd.to_numeric(df['lmp_year_full'], errors='coerce').astype(float),
    'month': pd.to_numeric(df['lmp_month_raw'], errors='coerce').astype(float),
    'day': pd.to_numeric(df['lmp_day_raw'], errors='coerce').astype(float),
}, index=df.index)
df['lmp_date'] = pd.to_datetime(lmp_components, errors='coerce')
df['lmp_to_birth_days'] = (df['birth_date'] - df['lmp_date']).dt.days
df['lmp_gest_weeks'] = np.floor(df['lmp_to_birth_days'] / 7)
df['lmp_exact_plausible'] = (
    df['lmp_date'].notna() & df['lmp_gest_weeks'].between(17, 52)
).astype(np.int64)
df['lmp_dgestat_diff_days'] = (df['lmp_date'] - df['gestation_start_dgestat']).dt.days
df['lmp_dgestat_abs_diff_days'] = df['lmp_dgestat_diff_days'].abs()

lmp_preferred = (
    df['lmp_exact_plausible'].eq(1)
    & (df['gestation_start_dgestat'].isna() | df['lmp_dgestat_abs_diff_days'].le(7))
)
df['gestation_start'] = df['gestation_start_dgestat']
df.loc[lmp_preferred, 'gestation_start'] = df.loc[lmp_preferred, 'lmp_date']
df['gestation_start_source'] = np.where(lmp_preferred, 'LMP', 'dgestat')
df.loc[df['gestation_start'].isna(), 'gestation_start_source'] = 'missing'
df['gestation_start_valid'] = (
    df['gestation_start'].notna() & df['birth_date'].notna()
    & df['gestation_start'].le(df['birth_date'])
).astype(np.int64)

print('Valid gestation starts:', int(df['gestation_start_valid'].sum()))
print(df['gestation_start_source'].value_counts(dropna=False))

# LMP/dgestat agreement QA.
print('\nLMP/dgestat absolute difference among records with both:')
print(df.loc[df['lmp_date'].notna() & df['gestation_start_dgestat'].notna(),
             'lmp_dgestat_abs_diff_days'].describe(percentiles=[.5,.75,.9,.95,.99]))


Valid gestation starts: 234974
gestation_start_source
LMP        204027
dgestat     30947
missing      6659
Name: count, dtype: int64

LMP/dgestat absolute difference among records with both:
count    204059.000000
mean          3.014555
std           4.919301
min           0.000000
50%           3.000000
75%           5.000000
90%           6.000000
95%           6.000000
99%           6.000000
max         366.000000
Name: lmp_dgestat_abs_diff_days, dtype: float64


In [ ]:
# ============================================================
# 3. ESTIMATED CONCEPTION AND FINAL T1 EXPOSURE WINDOW
# ============================================================
df['estimated_conception_date'] = df['gestation_start'] + pd.Timedelta(days=14)
df['conception_quarter'] = df['estimated_conception_date'].dt.quarter.astype('Int64')

# Final exposure window: gestational days 0 through 97 inclusive (98 days).
df['t1_start'] = df['gestation_start']
df['t1_end'] = df['gestation_start'] + pd.Timedelta(days=T1_END_DAY)


In [ ]:
# ============================================================
# 4. COUNTY CROSSWALK + WATER-PANEL BOUNDARIES
# ============================================================
df_fips = pd.read_csv(FIPS_PATH, low_memory=False)
df_fips['cntyres_clean'] = pd.to_numeric(df_fips['cntyres'], errors='coerce').astype('Int64')
df_fips['county_fips'] = pd.to_numeric(df_fips['fips'], errors='coerce').astype('Int64')
assert df_fips['cntyres_clean'].is_unique
county_crosswalk = df_fips[['cntyres_clean', 'county_fips', 'countyname']].copy()

df['cntyres_clean'] = numeric_column(df, 'cntyres').astype('Int64')
df = df.merge(county_crosswalk, on='cntyres_clean', how='left', validate='many_to_one')
df['county_mapped'] = df['county_fips'].notna().astype(np.int64)

water_panel = pd.read_csv(WATER_PANEL_PATH, low_memory=False)
water_panel['county_fips'] = pd.to_numeric(water_panel['county_fips'], errors='coerce').astype('Int64')
water_panel['quarter_start'] = pd.to_datetime(water_panel['quarter_start'], errors='coerce')
water_panel['quarter_end'] = pd.to_datetime(water_panel['quarter_end'], errors='coerce')
WATER_PANEL_START = water_panel['quarter_start'].min()
WATER_PANEL_END = water_panel['quarter_end'].max()
water_counties = set(water_panel['county_fips'].dropna().astype(int))
df['county_in_water_panel'] = df['county_fips'].isin(water_counties).astype(np.int64)
print('Water panel:', WATER_PANEL_START.date(), 'through', WATER_PANEL_END.date())


Water panel: 1982-01-01 through 1988-12-31


In [ ]:
# ============================================================
# 5. BIRTH MEASURES, PLAUSIBILITY, SINGLETON, AND INFANT SEX
# ============================================================
# Retain birthweight in grams; do not create LBW/VLBW outcomes here.
df['birthweight_raw'] = numeric_column(df, 'dbirwt')
df['birthweight_g'] = df['birthweight_raw'].where(df['birthweight_raw'].between(227, 8165))

# Gestational-age plausibility used only to define the linkage cohort.
valid_gest = df['gest_age_valid'].eq(1)
df['gest_20_41'] = np.where(
    valid_gest,
    df['gest_age_weeks'].between(PRIMARY_GEST_MIN_WEEKS, PRIMARY_GEST_MAX_COMPLETED_WEEKS).astype(int),
    np.nan
)

# Restrict final cohort to singleton births.
df['plurality_raw'] = numeric_column(df, 'dplural')
df['plurality'] = df['plurality_raw'].where(df['plurality_raw'].isin([1, 2, 3]))
df['singleton'] = np.select(
    [df['plurality'].eq(1), df['plurality'].isin([2, 3])],
    [1, 0],
    default=np.nan
)

# Infant sex: NCHS csex 1=male, 2=female.
df['infant_sex_raw'] = numeric_column(df, 'csex')
df['infant_male'] = np.select(
    [df['infant_sex_raw'].eq(1), df['infant_sex_raw'].eq(2)],
    [1, 0],
    default=np.nan
)


In [ ]:
# ============================================================
# 6. MATERNAL / INFANT COVARIATES NEEDED DOWNSTREAM
# ============================================================
df['maternal_age_raw'] = numeric_column(df, 'dmage')
df['maternal_age'] = df['maternal_age_raw'].where(df['maternal_age_raw'].between(10, 49))

df['live_birth_order_raw'] = numeric_column(df, 'dlivord')
df['live_birth_order'] = df['live_birth_order_raw'].where(df['live_birth_order_raw'].between(1, 50))

df['maternal_education_raw'] = numeric_column(df, 'dmeduc')
df['maternal_education_years'] = df['maternal_education_raw'].where(
    df['maternal_education_raw'].between(0, 17)
)

df['maternal_race_raw'] = numeric_column(df, 'mrace')
df['maternal_race_broad'] = np.select(
    [
        df['maternal_race_raw'].eq(1),
        df['maternal_race_raw'].eq(2),
        df['maternal_race_raw'].eq(3),
        df['maternal_race_raw'].isin([0, 4, 5, 6, 8]),
        df['maternal_race_raw'].eq(7),
    ],
    ['White', 'Black', 'American Indian', 'Asian/Pacific Islander', 'Other nonwhite'],
    default=None
)

df['marital_status_raw'] = numeric_column(df, 'dmar')
df['married'] = np.select(
    [df['marital_status_raw'].eq(1), df['marital_status_raw'].eq(2)],
    [1, 0],
    default=np.nan
)

df['prenatal_care_month_raw'] = numeric_column(df, 'monprec')
df['prenatal_care_month'] = df['prenatal_care_month_raw'].where(
    df['prenatal_care_month_raw'].between(1, 9)
)
prenatal_known = df['prenatal_care_month'].notna() | df['prenatal_care_month_raw'].eq(0)
df['prenatal_by5'] = np.where(
    prenatal_known,
    df['prenatal_care_month'].between(1, 5).fillna(False).astype(int),
    np.nan
)

print('\nMaternal race distribution:')
print(df['maternal_race_broad'].value_counts(dropna=False))
print('\nPrenatal-by-month-5 coding:')
print(pd.Series(df['prenatal_by5']).value_counts(dropna=False))



Maternal race distribution:
maternal_race_broad
White                     232822
Black                       5422
Asian/Pacific Islander      2538
American Indian              729
None                         113
Other nonwhite                 9
Name: count, dtype: int64

Prenatal-by-month-5 coding:
prenatal_by5
1.0    230348
0.0      9602
NaN      1683
Name: count, dtype: int64


In [ ]:
# ============================================================
# 7. GESTATIONAL-START COHORT + ADMINISTRATIVE FOLLOW-UP
# ============================================================
RAW_BIRTH_DATA_END = df['birth_date'].max()
ADMIN_GEST_START_CUTOFF = RAW_BIRTH_DATA_END - pd.Timedelta(days=MAX_FOLLOWUP_DAYS)
print('Birth-data end:', RAW_BIRTH_DATA_END.date())
print('Latest eligible gestational start for complete 41w6d follow-up:', ADMIN_GEST_START_CUTOFF.date())

base_valid = (
    df['birth_date_valid'].eq(1)
    & df['county_mapped'].eq(1)
    & df['county_in_water_panel'].eq(1)
    & df['gestation_start_valid'].eq(1)
)
df['cohort_start_ok'] = df['gestation_start'].ge(STUDY_GEST_START).astype(np.int64)
df['cohort_end_ok'] = df['gestation_start'].le(ADMIN_GEST_START_CUTOFF).astype(np.int64)
df['gest_plausible_20_41'] = df['gest_20_41'].fillna(0).astype(np.int64)

# Only the final T1 window must be inside the water panel.
df['t1_window_in_panel'] = (
    df['t1_start'].ge(WATER_PANEL_START)
    & df['t1_end'].le(WATER_PANEL_END)
).astype(np.int64)

df['linkage_cohort'] = (
    base_valid
    & df['cohort_start_ok'].eq(1)
    & df['cohort_end_ok'].eq(1)
    & df['t1_window_in_panel'].eq(1)
    & pd.Series(df['singleton']).eq(1)
    & df['gest_plausible_20_41'].eq(1)
).astype(np.int64)

sample_counts = pd.DataFrame({
    'sample': [
        'births_delivered_1983_1988',
        'valid_birth_county_gestation_start',
        'gestational_start_cohort_with_complete_followup',
        't1_window_inside_water_panel',
        'final_linkage_cohort_singleton_ga20_41',
    ],
    'n': [
        len(df),
        int(base_valid.sum()),
        int((base_valid & df['cohort_start_ok'].eq(1) & df['cohort_end_ok'].eq(1)).sum()),
        int((base_valid & df['cohort_start_ok'].eq(1) & df['cohort_end_ok'].eq(1) & df['t1_window_in_panel'].eq(1)).sum()),
        int(df['linkage_cohort'].sum()),
    ]
})
display(sample_counts)


Birth-data end: 1988-12-31
Latest eligible gestational start for complete 41w6d follow-up: 1988-03-13


,sample,n
0,births_delivered_1983_1988,241633
1,valid_birth_county_gestation_start,234974
2,gestational_start_cohort_with_complete_followup,200406
3,t1_window_inside_water_panel,200406
4,final_linkage_cohort_singleton_ga20_41,164977


In [ ]:
# ============================================================
# 8. COHORT-EDGE AND SAMPLE-FLOW QA
# ============================================================
df['gest_start_year'] = df['gestation_start'].dt.year.astype('Int64')
df['gest_start_quarter'] = df['gestation_start'].dt.quarter.astype('Int64')

flow_by_gest_start = (
    df.loc[df['gestation_start'].notna()]
    .groupby(['gest_start_year', 'gest_start_quarter'], as_index=False)
    .agg(
        births=('birth_id', 'size'),
        linkage_cohort=('linkage_cohort', 'sum')
    )
)
flow_by_gest_start['pct_linkage'] = (
    100 * flow_by_gest_start['linkage_cohort'] / flow_by_gest_start['births']
)
display(flow_by_gest_start)

flow_by_birth_year = (
    df.groupby('birth_year', as_index=False)
    .agg(
        births=('birth_id', 'size'),
        linkage_cohort=('linkage_cohort', 'sum')
    )
)
flow_by_birth_year['pct_linkage'] = (
    100 * flow_by_birth_year['linkage_cohort'] / flow_by_birth_year['births']
)
display(flow_by_birth_year)

# Source of gestational start among the final linkage cohort.
gest_start_source_qa = (
    df.loc[df['linkage_cohort'].eq(1), 'gestation_start_source']
    .value_counts(dropna=False)
    .rename_axis('gestation_start_source')
    .reset_index(name='n')
)
gest_start_source_qa['percent'] = 100 * gest_start_source_qa['n'] / gest_start_source_qa['n'].sum()
display(gest_start_source_qa)


,gest_start_year,gest_start_quarter,births,linkage_cohort,pct_linkage
0,1982,1,999,0,0.000000
1,1982,2,10132,0,0.000000
2,1982,3,10732,0,0.000000
3,1982,4,11157,0,0.000000
4,1983,1,9817,7974,81.226444
5,1983,2,10114,8204,81.115286
6,1983,3,10076,8095,80.339420
7,1983,4,11287,9227,81.748915
8,1984,1,9782,7963,81.404621
9,1984,2,10141,8252,81.372646


,birth_year,births,linkage_cohort,pct_linkage
0,1983,43258,7951,18.380415
1,1984,42366,33576,79.252231
2,1985,41221,32799,79.568666
3,1986,38771,31325,80.794924
4,1987,37898,30387,80.181012
5,1988,38119,28939,75.917521


,gestation_start_source,n,percent
0,LMP,143725,87.118204
1,dgestat,21252,12.881796


In [ ]:
# ============================================================
# 9. SAVE COMPACT LINKAGE INPUT + QA
# ============================================================
# Keep only fields required for T1 linkage and final covariate construction.
keep_cols = [
    'birth_id',
    'county_fips',
    'birth_year',
    'gest_age_weeks',
    'conception_quarter',
    'birthweight_g',
    'infant_male',
    'maternal_age',
    'maternal_race_broad',
    'married',
    'prenatal_by5',
    'live_birth_order',
    'maternal_education_years',
    't1_start',
    't1_end',
]

births_linkage = df.loc[df['linkage_cohort'].eq(1), keep_cols].copy()

assert births_linkage['birth_id'].is_unique
assert births_linkage['county_fips'].notna().all()
assert births_linkage['gest_age_weeks'].between(20, 41).all()
assert births_linkage['t1_start'].notna().all()
assert births_linkage['t1_end'].notna().all()
assert ((births_linkage['t1_end'] - births_linkage['t1_start']).dt.days == 97).all()

LINK_CSV = os.path.join(OUTPUT_DIR, 'births_linkage_T1_v3.csv')
LINK_PARQUET = os.path.join(OUTPUT_DIR, 'births_linkage_T1_v3.parquet')
births_linkage.to_csv(LINK_CSV, index=False)
try:
    births_linkage.to_parquet(LINK_PARQUET, index=False)
except Exception as exc:
    print('Parquet save skipped:', exc)

sample_counts.to_csv(os.path.join(OUTPUT_DIR, 'birth_sample_counts_T1_v3.csv'), index=False)
flow_by_gest_start.to_csv(os.path.join(OUTPUT_DIR, 'birth_flow_by_gestational_start_T1_v3.csv'), index=False)
flow_by_birth_year.to_csv(os.path.join(OUTPUT_DIR, 'birth_flow_by_birth_year_T1_v3.csv'), index=False)
gest_start_source_qa.to_csv(os.path.join(OUTPUT_DIR, 'gestation_start_source_qa_T1_v3.csv'), index=False)

metadata = pd.DataFrame({
    'item': [
        'study_gestational_start',
        'birth_data_end',
        'max_primary_completed_gest_weeks',
        'max_followup_days',
        'admin_gest_start_cutoff',
        'race_restriction_during_prep',
        'final_exposure_window',
        'final_linkage_population',
    ],
    'value': [
        str(STUDY_GEST_START.date()),
        str(RAW_BIRTH_DATA_END.date()),
        PRIMARY_GEST_MAX_COMPLETED_WEEKS,
        MAX_FOLLOWUP_DAYS,
        str(ADMIN_GEST_START_CUTOFF.date()),
        'None; all maternal race groups retained',
        'T1 = gestational days 0-97 inclusive (98 days)',
        'Singleton births, gestational age 20-41 completed weeks, gestational-start cohort with outcome-independent right-edge follow-up',
    ]
})
metadata.to_csv(os.path.join(OUTPUT_DIR, 'birth_prep_metadata_T1_v3.csv'), index=False)

print('File 2 Final V3 complete.')
print('Linkage input for File 3:', LINK_CSV)


File 2 Final V3 complete.
Linkage input for File 3: /content/drive/MyDrive/plos-update-v3/2-birth/births_linkage_T1_v3.csv
